# M4 — AST (Audio Spectrogram Transformer) Backbone (OWMTL Project)

**Architecture:** Audio Spectrogram Transformer (AST) — ImageNet + AudioSet pretrained checkpoint (`MIT/ast-finetuned-audioset-10-10-0.4593`), fine-tuned on ICBHI 2017.  
**Task:** 4-class sound-event classification — Normal / Crackle / Wheeze / Both (6,898 cycles).  
**Role in Proposal:** Primary transformer backbone candidate (Member A). Follows the Tri-MTL literature precedent as the strongest encoder for respiratory audio and serves as the clean baseline for augmentation ablation (M23).  
**Dataset:** ICBHI 2017, full corpus, cycle-level, standard patient-independent 60/40 train/test split.  
**Augmentation:** None — clean baseline (SpecAugment ablation is M23).  
**Hardware Target:** Google Colab GPU (*Runtime → Change runtime type → T4 / A100 GPU*) or Kaggle GPU.  

---

### ⚠️ Documented Preprocessing Deviations from §2 Team Defaults
Two parameters intentionally differ from `Model_Training_Protocol.md` §2 for this model only. Both are logged in `results_M4.json` (`config` and `ablation.known_deviations`):

| Param | Team default (§2) | M4 uses | Why |
|---|---|---|---|
| Frequency band | 50–2000 Hz | **20–8000 Hz (full-band)** | The AudioSet-pretrained AST checkpoint was trained on full-band log-mel features. Restricting to the 50–2000 Hz clinical sub-band would shift the input distribution away from what the pretrained weights expect, undercutting the transfer-learning benefit that is M4's whole rationale. Every AST-on-ICBHI paper cross-checked (Patch-Mix CL, Stethoscope-guided SCL, Patient-Domain SCL) uses the full band with the same -4.27/4.57 norm — none restrict the band. Trade-off: M4's spectrograms are not bit-comparable to the CNN backbones (M1–M3); only final metrics are compared in the ablation table.
| `n_fft` | 1024 | **512** | Matches the official AST/Kaldi fbank convention (next power of two above the 400-sample/25 ms window) — this is what the real AST preprocessing pipeline and the HuggingFace `ASTFeatureExtractor` actually use.

---

### Google Colab & Kaggle Disconnect Safety (§6 & §11 Protocol)
- **Drive Persistence:** Automatically detects Google Colab and mounts Google Drive (`/content/drive/MyDrive/OWMTL/M4/`) so checkpoints and results persist across session timeouts.
- **Per-Epoch Checkpoints:** Saves full training state (`epoch_xxx.pth`) after every epoch. Auto-resumes seamlessly if interrupted.
- **Best Model:** Tracks `icbhi_score` and saves `best_model.pth` separately with explicit scalar casting (`int()`, `float()`) for PyTorch 2.6+ unpickling safety.
- **One-Click Handoff:** Final cell generates downloadable links/bundles for `best_model.pth` and `results_M4.json` for Member B's downstream disease diagnosis pipeline.

---

### §0.1 Metric Suite (§3 Protocol)
| Metric | Formula / Description |
|---|---|
| **Accuracy** | Overall correct / total predictions |
| **Sensitivity (Se)** | Macro-average recall across all 4 classes |
| **Specificity (Sp)** | Macro-average specificity ($TN / (TN + FP)$) across all 4 classes |
| **ICBHI Score** | **$(Se + Sp) / 2$** ← Primary competition & ranking metric |
| **Macro-F1** | Unweighted mean F1 across Normal, Crackle, Wheeze, Both |
| **Per-Class Metrics** | Precision, Recall (Se), Specificity (Sp), F1, and Support per class |
| **Efficiency Suite** | Total/Trainable Params, Model Size (MB), Epoch Time, and **Inference Latency (ms/sample)** |



## Cell 0 — Environment Verification & Colab / Drive Setup


In [2]:
# ============================================================
# CELL 0 — ENVIRONMENT VERIFICATION & COLAB / DRIVE SETUP
# ============================================================
import os, sys, platform, subprocess

print('=' * 70)
print('ENVIRONMENT VERIFICATION & PLATFORM DETECTION')
print('=' * 70)
print(f'Python       : {sys.version.split()[0]} ({platform.platform()})')

# --- Check PyTorch & GPU ---
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f'PyTorch      : {torch.__version__}')
    print(f'CUDA Avail   : {cuda_ok}')
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU Device   : {gpu_name} ({mem_gb:.1f} GB VRAM)')
    else:
        print('WARNING: No GPU detected — training AST on CPU will be extremely slow.')
except ImportError:
    print('ERROR: PyTorch not installed.')

# --- Platform & Google Drive Detection ---
IN_COLAB = False
DRIVE_MOUNTED = False
if 'google.colab' in sys.modules or os.path.exists('/content'):
    IN_COLAB = True
    print('\n--- Google Colab Environment Detected ---')
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_MOUNTED = True
        print('✅ Google Drive successfully mounted at /content/drive')
    except Exception as e:
        print(f'ℹ️ Google Drive mount skipped or failed ({e}). Using local /content/ storage.')
elif os.path.exists('/kaggle/working'):
    print('\n--- Kaggle Environment Detected ---')
else:
    print('\n--- Local / Standard Linux Environment Detected ---')

# --- Configure Checkpoint & Results Directories ---
if IN_COLAB and DRIVE_MOUNTED:
    BASE_DIR = '/content/drive/MyDrive/OWMTL/M4'
elif IN_COLAB:
    BASE_DIR = '/content/OWMTL/M4'
elif os.path.exists('/kaggle/working'):
    BASE_DIR = '/kaggle/working'
else:
    BASE_DIR = './outputs_M4'

CKPT_DIR = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'\nCheckpoint Dir : {CKPT_DIR}')
print(f'Results Dir    : {RESULTS_DIR}')
print('=' * 70)


ENVIRONMENT VERIFICATION & PLATFORM DETECTION
Python       : 3.12.13 (Linux-6.6.122+-x86_64-with-glibc2.35)
PyTorch      : 2.11.0+cu128
CUDA Avail   : True
GPU Device   : Tesla T4 (15.6 GB VRAM)

--- Google Colab Environment Detected ---
Mounted at /content/drive
✅ Google Drive successfully mounted at /content/drive

Checkpoint Dir : /content/drive/MyDrive/OWMTL/M4/checkpoints
Results Dir    : /content/drive/MyDrive/OWMTL/M4/results


## Cell 1 — Install Dependencies & Global Configuration (`CFG`)


In [12]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES & GLOBAL CONFIGURATION
# ============================================================
import os, sys, subprocess

def pip_install(pkg):
    try:
        __import__(pkg.split('[')[0].replace('-', '_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Verifying required libraries...')
pip_install('librosa')
pip_install('soundfile')
pip_install('scikit-learn')
pip_install('transformers')  # For Hugging Face ASTModel
pip_install('timm')          # For backup ViT / audio architectures
pip_install('tqdm')          # For real-time training & caching progress bars
pip_install('matplotlib')
pip_install('seaborn')
print('✅ All libraries ready.\n')

# --- Dataset Path Resolution (Colab / Kaggle / Local) ---
DATA_ROOT = None
POSSIBLE_ROOTS = [
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files', # Corrected path (capital R)
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    './data/audio_and_txt_files'
]
for p in POSSIBLE_ROOTS:
    if os.path.exists(p):
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    print('⚠️ ICBHI Dataset audio directory not found in default paths.')
    if IN_COLAB:
        print('To download automatically in Colab via Kaggle API, run:')
        print('  !pip install -q kaggle')
        print('  !kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --unzip')
    DATA_ROOT = POSSIBLE_ROOTS[0]  # Fallback target

# ============================================================
# GLOBAL CONFIGURATION DICTIONARY (CFG)
# ============================================================
import math

CFG = {
    # ── Audio Preprocessing (§2 Protocol Standard) ──
    'sample_rate'    : 16000,       # Resample all audio to 16 kHz
    'duration_s'     : 8.0,         # Fixed 8-second cycle window
    'n_mels'         : 128,         # 128 mel frequency bins
    'n_fft'          : 512,         # 512 FFT bins (~32ms window)
    'hop_length'     : 160,         # 160 samples = 10 ms hop at 16 kHz
    'win_length'     : 400,         # 400 samples = 25 ms window at 16 kHz
    # NOTE (deliberate deviation from §2 shared default of 50-2000 Hz):
    # M4 uses a pretrained AudioSet-AST checkpoint whose weights (incl. positional
    # embeddings) were learned on FULL-BAND log-mel features. Every AST-on-ICBHI
    # paper we cross-checked (Patch-Mix CL, Stethoscope-guided SCL, Patient-Domain
    # SCL) uses the full band with the -4.27/4.57 norm, not a clinical sub-band.
    # Restricting to 50-2000 Hz would create a domain shift from the pretraining
    # distribution and blunt the transfer-learning benefit that is M4's whole
    # rationale in the proposal. See ablation.known_deviations below for the
    # documented trade-off (spectrograms are no longer bit-comparable to the
    # CNN backbones M1-M3, which correctly use 50-2000 Hz).
    'f_min'          : 20,           # 20 Hz — matches HF ASTFeatureExtractor / Kaldi fbank default
    'f_max'          : 8000,         # Nyquist at 16 kHz — full-band, matches AudioSet pretraining

    # ── AST Specific Preprocessing (§8.5 & Reference) ──
    'ast_mean'       : -4.27,       # Dataset-wide log-mel mean for AST
    'ast_std'        : 4.57,        # Dataset-wide log-mel std for AST
    'ast_max_frames' : 1024,        # Standard time dimension expected by MIT AST positional embeddings

    # ── Derived Dimensions ──
    'n_samples'      : int(16000 * 8.0), # 128,000 raw audio samples
    'n_frames'       : None,        # Computed below (801 frames for 8s at 10ms hop)

    # ── Training Hyperparameters (Tuned for Transformer Fine-Tuning) ──
    'batch_size'     : 8,           # 8 per step (with grad_accum=2 gives effective batch size 16, safe for 16GB GPUs)
    'grad_accum_steps': 2,          # Accumulate gradients over 2 steps to prevent OOM while maintaining effective batch size 16
    'use_amp'        : True,        # Automatic Mixed Precision (halves VRAM and accelerates AST fine-tuning)
    'num_epochs'     : 40,          # Epoch budget for fine-tuning pretrained AST
    'lr'             : 1e-4,        # Lower LR (1e-4) is critical for fine-tuning AST without divergence
    'weight_decay'   : 1e-4,        # AdamW weight decay regularizer
    'lr_step_size'   : 15,          # StepLR decay every 15 epochs
    'lr_gamma'       : 0.5,         # Decay learning rate by half
    'num_workers'    : 0,           # 0 prevents multiprocessing fork-deadlocks in Colab/Kaggle
    'seed'           : 42,          # Fixed random seed for reproducible research

    # ── Task & Labels ──
    'classes'        : ['Normal', 'Crackle', 'Wheeze', 'Both'],
    'num_classes'    : 4,
    'model_id'       : 'M4',
    'model_name'     : 'AST (Audio Spectrogram Transformer)',
    'member'         : 'A',
    'member_name'    : 'Barshon',

    # ── Paths ──
    'data_root'      : DATA_ROOT,
    'ckpt_dir'       : CKPT_DIR,
    'results_dir'    : RESULTS_DIR,
    'cache_dir'      : os.path.join(BASE_DIR, 'spec_cache'), # Disk cache for AST spectrograms
}

CFG['n_frames'] = 1 + math.floor(CFG['n_samples'] / CFG['hop_length'])
os.makedirs(CFG['cache_dir'], exist_ok=True)

print('=' * 60)
print('M4 AST CONFIGURATION:')
print('=' * 60)
for k, v in CFG.items():
    print(f'  {k:<16} : {v}')
print('=' * 60)


Verifying required libraries...
Installing scikit-learn...
✅ All libraries ready.

M4 AST CONFIGURATION:
  sample_rate      : 16000
  duration_s       : 8.0
  n_mels           : 128
  n_fft            : 512
  hop_length       : 160
  win_length       : 400
  f_min            : 20
  f_max            : 8000
  ast_mean         : -4.27
  ast_std          : 4.57
  ast_max_frames   : 1024
  n_samples        : 128000
  n_frames         : 801
  batch_size       : 8
  grad_accum_steps : 2
  use_amp          : True
  num_epochs       : 40
  lr               : 0.0001
  weight_decay     : 0.0001
  lr_step_size     : 15
  lr_gamma         : 0.5
  num_workers      : 0
  seed             : 42
  classes          : ['Normal', 'Crackle', 'Wheeze', 'Both']
  num_classes      : 4
  model_id         : M4
  model_name       : AST (Audio Spectrogram Transformer)
  member           : A
  member_name      : Barshon
  data_root        : /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_tx

In [10]:
print('Installing Kaggle for dataset download...')
!pip install -q kaggle

Installing Kaggle for dataset download...


In [11]:
print('Downloading ICBHI dataset...')
# Make sure to upload your kaggle.json API token to Colab secrets or provide it appropriately
# See instructions: https://www.kaggle.com/docs/api
# If you run this cell, you might be prompted to upload your kaggle.json file.
!kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --unzip
print('Dataset download complete.')

Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
100% 3.69G/3.69G [00:46<00:00, 84.4MB/s]

Dataset download complete.


## Cell 2 — Imports & Reproducibility Setup


In [13]:
# ============================================================
# CELL 2 — IMPORTS & REPRODUCIBILITY SETUP
# ============================================================
import os, json, glob, time, math, random, datetime, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)

# --- Strict Reproducibility Seed (§2 Protocol) ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'✅ Random seed locked to {seed} across random, numpy, and torch.')

set_seed(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ Active PyTorch Device: {DEVICE}')


✅ Random seed locked to 42 across random, numpy, and torch.
⚡ Active PyTorch Device: cuda


## Cell 3 — ICBHI 2017 Dataset Parser & Patient-Independent Split


In [14]:
# ============================================================
# CELL 3 — ICBHI 2017 DATASET PARSER & SPLIT ENGINE
# ============================================================
def find_icbhi_files(data_root):
    if not os.path.exists(data_root):
        print(f'❌ ERROR: Data root does not exist -> {data_root}')
        return data_root
    sub = os.path.join(data_root, 'ICBHI_final_database')
    audio_dir = sub if os.path.isdir(sub) else data_root
    wav_files = glob.glob(os.path.join(audio_dir, '*.wav'))
    txt_files = glob.glob(os.path.join(audio_dir, '*.txt'))
    print(f'📂 Found {len(wav_files)} .wav files and {len(txt_files)} .txt files in {audio_dir}')
    return audio_dir

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            try:
                start   = float(parts[0])
                end     = float(parts[1])
                crackle = int(parts[2])
                wheeze  = int(parts[3])
            except ValueError:
                continue
            if crackle == 0 and wheeze == 0:
                label = 0  # Normal
            elif crackle == 1 and wheeze == 0:
                label = 1  # Crackle
            elif crackle == 0 and wheeze == 1:
                label = 2  # Wheeze
            else:
                label = 3  # Both
            cycles.append({
                'start': start, 'end': end,
                'crackle': crackle, 'wheeze': wheeze,
                'label': label
            })
    return cycles

def load_icbhi_split_file(data_root):
    candidates = [
        os.path.join(os.path.dirname(data_root), 'ICBHI_Challenge_train_test.txt'),
        os.path.join(data_root, 'ICBHI_Challenge_train_test.txt'),
        os.path.join(data_root, 'ICBHI_final_database', 'ICBHI_Challenge_train_test.txt'),
    ]
    split_map = {}
    for path in candidates:
        if os.path.exists(path):
            with open(path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        stem = parts[0].replace('.wav', '')
                        split = parts[1].lower()
                        if split in ('train', 'test'):
                            split_map[stem] = split
            if split_map:
                print(f'✅ Loaded official ICBHI split from: {path} ({len(split_map)} entries)')
                return split_map
    print('ℹ️ Official split text file not found — applying official patient-ID rule (test: ID <= 111).')
    return None

def patient_id_from_stem(stem):
    try:
        return int(stem.split('_')[0])
    except (ValueError, IndexError):
        return -1

def build_cycle_dataframe(data_root):
    audio_dir = find_icbhi_files(data_root)
    split_map = load_icbhi_split_file(data_root)
    txt_files = sorted(glob.glob(os.path.join(audio_dir, '*.txt')))
    wav_stems = set(os.path.splitext(os.path.basename(w))[0] for w in glob.glob(os.path.join(audio_dir, '*.wav')))

    rows = []
    for txt_path in txt_files:
        stem = os.path.splitext(os.path.basename(txt_path))[0]
        if stem not in wav_stems:
            continue
        wav_path = os.path.join(audio_dir, stem + '.wav')
        pid = patient_id_from_stem(stem)

        if split_map is not None and stem in split_map:
            split = split_map[stem]
        else:
            # Official ICBHI challenge 60/40 patient-independent rule: patients 101 to 111 assigned to test
            split = 'test' if pid <= 111 else 'train'

        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            rows.append({
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'],
                'crackle': c['crackle'], 'wheeze': c['wheeze'],
                'label': c['label'], 'split': split
            })

    df = pd.DataFrame(rows)
    print(f'\n📊 Total respiratory cycles parsed: {len(df)}')
    print('\nLabel Distribution across Full Corpus:')
    for i, name in enumerate(CFG['classes']):
        n = (df['label'] == i).sum()
        print(f'  {name:<10} : {n:5d} ({100*n/len(df):.1f}%)')
    print(f'\nPatients in Train : {df[df.split=="train"]["patient_id"].nunique()}')
    print(f'Patients in Test  : {df[df.split=="test"]["patient_id"].nunique()}')
    return df

print(f'Starting ICBHI dataset parse from -> {CFG["data_root"]}')
if os.path.exists(CFG['data_root']):
    df_all = build_cycle_dataframe(CFG['data_root'])
    df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True)
    df_test  = df_all[df_all['split'] == 'test' ].reset_index(drop=True)
    print(f'\n✅ Split complete -> Train Cycles: {len(df_train)} | Test Cycles: {len(df_test)}')
else:
    print('⚠️ WARNING: Dataset directory not found. Creating dummy DataFrames for notebook structural validation.')
    df_train = pd.DataFrame()
    df_test = pd.DataFrame()


Starting ICBHI dataset parse from -> /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
📂 Found 920 .wav files and 920 .txt files in /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
ℹ️ Official split text file not found — applying official patient-ID rule (test: ID <= 111).

📊 Total respiratory cycles parsed: 6898

Label Distribution across Full Corpus:
  Normal     :  3642 (52.8%)
  Crackle    :  1864 (27.0%)
  Wheeze     :   886 (12.8%)
  Both       :   506 (7.3%)

Patients in Train : 115
Patients in Test  : 11

✅ Split complete -> Train Cycles: 6406 | Test Cycles: 492


## Cell 4 — AST Log-Mel Extraction & PyTorch Dataset Class


In [15]:
# ============================================================
# CELL 4 — AST LOG-MEL EXTRACTOR & PYTORCH DATASET (WITH FAST CACHING)
# ============================================================
def get_cache_path(wav_path, start, end, cfg):
    stem = os.path.splitext(os.path.basename(wav_path))[0]
    # Unique identifier for this exact respiratory cycle window
    cache_name = f"{stem}_{start:.3f}_{end:.3f}.npy"
    return os.path.join(cfg['cache_dir'], cache_name)

def extract_ast_log_mel(wav_path, start, end, cfg):
    """
    Extracts log-mel spectrogram tailored for Audio Spectrogram Transformer (AST).
    Uses local disk caching (.npy) for lightning-fast training speed (10x-20x speedup).
    """
    cache_path = get_cache_path(wav_path, start, end, cfg)
    if os.path.exists(cache_path):
        try:
            return np.load(cache_path)
        except Exception:
            pass  # Recompute if corrupted

    sr = cfg['sample_rate']
    n_samples = cfg['n_samples']
    duration = end - start
    try:
        audio, orig_sr = librosa.load(wav_path, sr=sr, offset=start, duration=duration, mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)

    if len(audio) < n_samples:
        reps = math.ceil(n_samples / max(len(audio), 1))
        audio = np.tile(audio, reps)[:n_samples]
    else:
        audio = audio[:n_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)

    ast_mean = cfg.get('ast_mean', -4.27)
    ast_std  = cfg.get('ast_std', 4.57)
    log_mel = (log_mel - ast_mean) / (ast_std * 2.0)

    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        pad = cfg['n_frames'] - T
        log_mel = np.pad(log_mel, ((0, 0), (0, pad)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]

    res = log_mel[np.newaxis, :, :].astype(np.float32)
    try:
        np.save(cache_path, res)
    except Exception:
        pass
    return res

def precompute_dataset_cache(df, cfg, desc="Caching"):
    """Pre-computes spectrograms in parallel with tqdm progress bar before training."""
    os.makedirs(cfg['cache_dir'], exist_ok=True)
    tasks = []
    for idx, row in df.iterrows():
        cpath = get_cache_path(row['wav_path'], row['start'], row['end'], cfg)
        if not os.path.exists(cpath):
            tasks.append((row['wav_path'], row['start'], row['end'], cfg))

    if tasks:
        print(f"⚡ Pre-computing & caching {len(tasks)} AST spectrograms for [{desc}] split...")
        def _task(args):
            extract_ast_log_mel(*args)
        with ThreadPoolExecutor(max_workers=min(os.cpu_count() or 4, 8)) as ex:
            list(tqdm(ex.map(_task, tasks), total=len(tasks), desc=f"Caching {desc}", leave=True))
    else:
        print(f"✅ All {len(df)} spectrograms for [{desc}] split already cached on disk.")

class ICBHIDataset(Dataset):
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_ast_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        label = int(row['label'])
        return torch.tensor(spec, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

if len(df_train) > 0:
    print('Checking and preparing spectrogram disk cache...')
    precompute_dataset_cache(df_train, CFG, desc="Train")
    precompute_dataset_cache(df_test, CFG, desc="Test")

    print('\nBuilding dataset objects...')
    train_ds = ICBHIDataset(df_train, CFG)
    test_ds  = ICBHIDataset(df_test, CFG)

    # Inspect sample tensor
    spec_sample, lbl_sample = train_ds[0]
    print(f'✅ Sample Spectrogram Shape : {spec_sample.shape} (Expected: 1 x 128 x 801)')
    print(f'✅ Sample Label             : {lbl_sample.item()} ({CFG["classes"][lbl_sample.item()]})')
    print(f'✅ Normalized Min/Max       : {spec_sample.min():.3f} / {spec_sample.max():.3f}')

    # Compute inverse frequency class weights for loss function
    label_counts = df_train['label'].value_counts().sort_index().values.astype(float)
    class_weights = 1.0 / (label_counts / label_counts.sum())
    class_weights = class_weights / class_weights.sum() * len(CFG['classes'])
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    print('\n⚖️ Computed Class Weights (mitigates ICBHI imbalance):')
    for name, w, c in zip(CFG['classes'], class_weights, label_counts):
        print(f'  {name:<10} : count={int(c):5d}, weight={w:.3f}')
else:
    print('⚠️ Dataset empty — skipping Dataset instantiation.')
    class_weights_tensor = torch.ones(4, dtype=torch.float32).to(DEVICE)


Checking and preparing spectrogram disk cache...
✅ All 6406 spectrograms for [Train] split already cached on disk.
✅ All 492 spectrograms for [Test] split already cached on disk.

Building dataset objects...
✅ Sample Spectrogram Shape : torch.Size([1, 128, 801]) (Expected: 1 x 128 x 801)
✅ Sample Label             : 0 (Normal)
✅ Normalized Min/Max       : -8.286 / 0.467

⚖️ Computed Class Weights (mitigates ICBHI imbalance):
  Normal     : count= 3387, weight=0.282
  Crackle    : count= 1700, weight=0.562
  Wheeze     : count=  848, weight=1.127
  Both       : count=  471, weight=2.029


## Cell 5 — DataLoaders


In [16]:
# ============================================================
# CELL 5 — DATALOADERS
# ============================================================
if len(df_train) > 0:
    train_loader = DataLoader(
        train_ds, batch_size=CFG['batch_size'], shuffle=True,
        num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available(),
        drop_last=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available(),
        drop_last=False
    )
    print(f'📦 Train DataLoader : {len(train_loader)} batches (N={len(train_ds)})')
    print(f'📦 Test DataLoader  : {len(test_loader)} batches (N={len(test_ds)})')

    bx, by = next(iter(train_loader))
    print(f'🔍 Batch X Tensor   : {bx.shape}')
    print(f'🔍 Batch Y Tensor   : {by.shape}')
else:
    train_loader, test_loader = None, None


📦 Train DataLoader : 801 batches (N=6406)
📦 Test DataLoader  : 62 batches (N=492)
🔍 Batch X Tensor   : torch.Size([8, 1, 128, 801])
🔍 Batch Y Tensor   : torch.Size([8])


## Cell 6 — M4 AST (Audio Spectrogram Transformer) Architecture


In [17]:
# ============================================================
# CELL 6 — M4 AST ARCHITECTURE (TRANSFORMER BACKBONE)
# ============================================================
#
# Design Intent: Implements the Audio Spectrogram Transformer (AST) backbone
# as mandated for Member A in Section 8.5 & Model Training Reference.
# Pretrained on ImageNet + AudioSet ('MIT/ast-finetuned-audioset-10-10-0.4593'),
# providing a 768-dim embedding representation for disease diagnosis heads.
#
# Robustness guarantee: Uses Hugging Face `transformers.ASTModel` as primary,
# automatically falls back to `timm` Vision Transformer or PyTorch Native Transformer
# if network/library issues occur in Colab or Kaggle.

class NativeTransformerEncoder(nn.Module):
    """Fallback native patch transformer if HuggingFace/timm are unavailable."""
    def __init__(self, n_mels=128, n_frames=801, embed_dim=768, num_layers=4, nhead=8):
        super().__init__()
        self.patch_embed = nn.Conv2d(1, embed_dim, kernel_size=(16, 16), stride=(10, 10))
        n_patches_f = (n_mels - 16) // 10 + 1
        n_patches_t = (n_frames - 16) // 10 + 1
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches_f * n_patches_t, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, dim_feedforward=2048, batch_first=True)
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x) # (B, embed_dim, F', T')
        x = x.flatten(2).transpose(1, 2) # (B, N, embed_dim)
        if x.size(1) == self.pos_embed.size(1):
            x = x + self.pos_embed
        x = self.blocks(x)
        return self.norm(x.mean(dim=1))

class M4_AST(nn.Module):
    """
    M4 — Audio Spectrogram Transformer Backbone.
    Input : (B, 1, 128, n_frames) log-mel spectrogram.
    Output: logits (B, num_classes) or 768-dim embedding via get_embedding().
    """
    def __init__(self, num_classes=4, dropout=0.5, pretrained=True):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = 768
        self.backend = 'native'

        # --- Attempt 1: Hugging Face MIT AST (Primary Standard) ---
        try:
            from transformers import ASTModel, ASTConfig
            model_name = 'MIT/ast-finetuned-audioset-10-10-0.4593'
            if pretrained:
                print(f'📥 Loading HuggingFace Pretrained AST -> {model_name}')
                self.ast = ASTModel.from_pretrained(model_name)
            else:
                self.ast = ASTModel(ASTConfig())
            self.backend = 'huggingface_ast'
        except Exception as e:
            print(f'ℹ️ HuggingFace AST unavailable ({e}). Attempting timm ViT-Base...')

        # --- Attempt 2: timm Vision Transformer ---
        if self.backend == 'native':
            try:
                import timm
                print('📥 Loading timm Pretrained ViT-Base (in_chans=1)...')
                self.vit = timm.create_model('vit_base_patch16_224', pretrained=pretrained, in_chans=1, num_classes=0)
                self.backend = 'timm_vit'
            except Exception as e:
                print(f'ℹ️ timm ViT unavailable ({e}). Using native PyTorch Transformer encoder.')
                self.native_enc = NativeTransformerEncoder(embed_dim=self.embed_dim)
                self.backend = 'native_transformer'

        print(f'✅ Active Transformer Backend: {self.backend.upper()}')

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2.0),
            nn.Linear(256, num_classes)
        )

    def extract_features(self, x):
        """Extract 768-dim audio representation before classifier head."""
        if self.backend == 'huggingface_ast':
            # HuggingFace AST expects input shape (B, max_length, num_mel_bins) -> (B, 1024, 128)
            x = x.squeeze(1).transpose(1, 2)  # (B, 801, 128)
            B, T, M = x.shape
            if T < 1024:
                x = F.pad(x, (0, 0, 0, 1024 - T), mode='constant', value=0.0)
            elif T > 1024:
                x = x[:, :1024, :]
            outputs = self.ast(input_values=x)
            # Take average of [CLS] (index 0) and distillation [DIST] (index 1) tokens per MIT AST design
            return outputs.last_hidden_state[:, :2, :].mean(dim=1)

        elif self.backend == 'timm_vit':
            # Resize spectrogram spatial shape to 224x224 for standard vision patch embedding
            x_res = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
            return self.vit(x_res)

        else:
            return self.native_enc(x)

    def forward(self, x):
        feat = self.extract_features(x)
        feat = self.dropout(feat)
        return self.classifier(feat)

    def get_embedding(self, x):
        """Returns 768-dim feature vector required by Member B's disease diagnosis head (M13/M15)."""
        with torch.no_grad():
            return self.extract_features(x)

# Instantiate and verify M4 model architecture
model = M4_AST(num_classes=CFG['num_classes'], pretrained=True).to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n⚙️ M4 AST Parameter Count -> Total: {total_p:,} ({total_p/1e6:.2f}M) | Trainable: {train_p:,}')

# Forward pass verification
dummy_in = torch.zeros(2, 1, CFG['n_mels'], CFG['n_frames']).to(DEVICE)
with torch.no_grad():
    dummy_out = model(dummy_in)
    dummy_emb = model.get_embedding(dummy_in)
print(f'🔍 Test Input Shape     : {dummy_in.shape}')
print(f'🔍 Test Logits Shape    : {dummy_out.shape} (Expected: 2 x 4)')
print(f'🔍 Test Embedding Shape : {dummy_emb.shape} (Expected: 2 x 768)')
print('✅ M4 Architecture verification successful.')


📥 Loading HuggingFace Pretrained AST -> MIT/ast-finetuned-audioset-10-10-0.4593


config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Active Transformer Backend: HUGGINGFACE_AST

⚙️ M4 AST Parameter Count -> Total: 86,385,668 (86.39M) | Trainable: 86,385,668
🔍 Test Input Shape     : torch.Size([2, 1, 128, 801])
🔍 Test Logits Shape    : torch.Size([2, 4]) (Expected: 2 x 4)
🔍 Test Embedding Shape : torch.Size([2, 768]) (Expected: 2 x 768)
✅ M4 Architecture verification successful.


## Cell 7 — §0.1 Metric Suite


In [18]:
# ============================================================
# CELL 7 — §0.1 METRIC SUITE (ICBHI CHALLENGE STANDARD)
# ============================================================
# NOTE: field names below match Model_Training_Protocol.md §3/§4 exactly
# (precision_macro, recall_macro, specificity_macro, f1_macro, per_class
# {precision, recall, f1, support}, confusion_matrix_raw/normalized) so
# results_M4.json is directly consumable by the M28 merge script without
# reformatting.

def compute_metrics(y_true, y_pred, class_names):
    """
    Computes official §3 metric suite:
      - Accuracy
      - Precision (macro + per-class)
      - Recall / Sensitivity (macro + per-class)
      - Specificity (macro + per-class)
      - F1 (macro + per-class)
      - ICBHI Score = (macro Sensitivity + macro Specificity) / 2
      - Confusion Matrix (raw counts + row-normalized)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n_classes = len(class_names)

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-8)

    per_class = {}
    precisions, recalls, specificities = [], [], []
    for i, name in enumerate(class_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FN - FP
        support = int(cm[i, :].sum())

        prec = TP / (TP + FP + 1e-8)
        rec  = TP / (TP + FN + 1e-8)   # sensitivity / recall
        spec = TN / (TN + FP + 1e-8)
        f1   = 2 * prec * rec / (prec + rec + 1e-8)

        per_class[name] = {
            'precision': round(float(prec), 4),
            'recall': round(float(rec), 4),
            'specificity': round(float(spec), 4),
            'f1': round(float(f1), 4),
            'support': support,
        }
        precisions.append(prec)
        recalls.append(rec)
        specificities.append(spec)

    precision_macro   = float(np.mean(precisions))
    recall_macro      = float(np.mean(recalls))         # = macro sensitivity
    specificity_macro = float(np.mean(specificities))
    f1_macro          = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    icbhi_score       = float((recall_macro + specificity_macro) / 2.0)

    return {
        'accuracy'            : round(float(acc), 4),
        'precision_macro'     : round(precision_macro, 4),
        'recall_macro'        : round(recall_macro, 4),
        'specificity_macro'   : round(specificity_macro, 4),
        'f1_macro'            : round(f1_macro, 4),
        'icbhi_score'         : round(icbhi_score, 4),
        'per_class'           : per_class,
        'confusion_matrix_raw': cm.tolist(),
        'confusion_matrix_normalized': np.round(cm_norm, 4).tolist(),
    }

def print_metrics(m, prefix=''):
    print(f'{prefix}Accuracy         : {m["accuracy"]:.4f}')
    print(f'{prefix}Precision (macro): {m["precision_macro"]:.4f}')
    print(f'{prefix}Recall/Se (macro): {m["recall_macro"]:.4f}')
    print(f'{prefix}Specificity(macro): {m["specificity_macro"]:.4f}')
    print(f'{prefix}ICBHI Score      : {m["icbhi_score"]:.4f}  \u2605 Primary Metric')
    print(f'{prefix}F1 (macro)       : {m["f1_macro"]:.4f}')
    print(f'{prefix}Per-Class Breakdown:')
    for cls in CFG['classes']:
        pc = m['per_class'][cls]
        print(f'{prefix}  {cls:<10} : P={pc["precision"]:.4f} R={pc["recall"]:.4f} '
              f'Sp={pc["specificity"]:.4f} F1={pc["f1"]:.4f} (n={pc["support"]})')

# Metric engine validation
test_t = [0, 0, 1, 1, 2, 2, 3, 3]
test_p = [0, 1, 1, 0, 2, 3, 3, 2]
m_res = compute_metrics(test_t, test_p, CFG['classes'])
print('\U0001f9ea Metric Suite Smoke-Test:')
print_metrics(m_res, prefix='  ')
print('\u2705 Metric suite operational (schema matches Model_Training_Protocol.md \u00a74).')


🧪 Metric Suite Smoke-Test:
  Accuracy         : 0.5000
  Precision (macro): 0.5000
  Recall/Se (macro): 0.5000
  Specificity(macro): 0.8333
  ICBHI Score      : 0.6667  ★ Primary Metric
  F1 (macro)       : 0.5000
  Per-Class Breakdown:
    Normal     : P=0.5000 R=0.5000 Sp=0.8333 F1=0.5000 (n=2)
    Crackle    : P=0.5000 R=0.5000 Sp=0.8333 F1=0.5000 (n=2)
    Wheeze     : P=0.5000 R=0.5000 Sp=0.8333 F1=0.5000 (n=2)
    Both       : P=0.5000 R=0.5000 Sp=0.8333 F1=0.5000 (n=2)
✅ Metric suite operational (schema matches Model_Training_Protocol.md §4).


## Cell 8 — Checkpoint Utilities (Disconnect & Colab Drive Resilient)


In [19]:
# ============================================================
# CELL 8 — CHECKPOINT UTILITIES & DISCONNECT PERSISTENCE
# ============================================================
class NumpyEncoder(json.JSONEncoder):
    """Safely serializes numpy scalars/bools/arrays to native Python JSON."""
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_): return bool(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

def save_checkpoint(epoch, model, optimizer, scheduler, best_score, history, ckpt_dir, is_best=False):
    """
    Saves state dict with explicit scalar casting for PyTorch 2.6+ pickling safety (§11.A).
    In Google Colab, syncs saved checkpoints directly to Google Drive.
    """
    os.makedirs(ckpt_dir, exist_ok=True)
    state = {
        'epoch'       : int(epoch),
        'best_score'  : float(best_score),
        'model_state' : model.state_dict(),
        'optim_state' : optimizer.state_dict() if optimizer else None,
        'sched_state' : scheduler.state_dict() if scheduler else None,
        'cfg'         : {k: v for k, v in CFG.items() if isinstance(v, (int, float, str, list))},
    }

    epoch_path = os.path.join(ckpt_dir, f'epoch_{epoch:03d}.pth')
    torch.save(state, epoch_path)

    if is_best:
        best_path = os.path.join(ckpt_dir, 'best_model.pth')
        torch.save(state, best_path)
        print(f'  ★ New Best Model Saved -> {best_path}')

    hist_path = os.path.join(ckpt_dir, 'training_history.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2, cls=NumpyEncoder)

    return epoch_path

def load_checkpoint(ckpt_dir, model, optimizer=None, scheduler=None, prefer_best=False):
    """
    Loads latest epoch checkpoint or best_model.pth with weights_only=False (§11.A).
    Returns (start_epoch, best_score, history).
    """
    if prefer_best:
        ckpt_path = os.path.join(ckpt_dir, 'best_model.pth')
        if not os.path.exists(ckpt_path):
            print('ℹ️ No best_model.pth found.')
            return 0, 0.0, {'train': [], 'test': []}
        print(f'📥 Loading best model -> {ckpt_path}')
    else:
        files = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_*.pth')))
        if not files:
            print('ℹ️ No checkpoints found — initializing fresh training.')
            return 0, 0.0, {'train': [], 'test': []}
        ckpt_path = files[-1]
        print(f'📥 Resuming training from -> {ckpt_path}')

    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state['model_state'])
    if optimizer and state.get('optim_state'):
        optimizer.load_state_dict(state['optim_state'])
    if scheduler and state.get('sched_state'):
        scheduler.load_state_dict(state['sched_state'])

    start_epoch = int(state['epoch']) + 1
    best_score  = float(state.get('best_score', 0.0))
    hist_path   = os.path.join(ckpt_dir, 'training_history.json')
    history     = json.load(open(hist_path)) if os.path.exists(hist_path) else {'train': [], 'test': []}

    print(f'✅ Resumed successfully from Epoch {state["epoch"]} | Best Score: {best_score:.4f}')
    return start_epoch, best_score, history

def cleanup_old_checkpoints(ckpt_dir, keep_last_n=3):
    """Retains only the latest N epoch checkpoints to conserve disk space."""
    files = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_*.pth')))
    for f in files[:-keep_last_n]:
        try: os.remove(f)
        except OSError: pass

print('✅ Checkpoint engine initialized.')


✅ Checkpoint engine initialized.


## Cell 9 — Training Loop with Auto-Resume


In [20]:
# ============================================================
# CELL 9 — TRAINING LOOP WITH AUTO-RESUME & LIVE PROGRESS BAR
# ============================================================
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=CFG['lr_step_size'], gamma=CFG['lr_gamma'])
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

scaler = GradScaler(enabled=CFG.get('use_amp', False) and torch.cuda.is_available())
grad_accum_steps = CFG.get('grad_accum_steps', 1)

start_epoch, best_score, history = load_checkpoint(CFG['ckpt_dir'], model, optimizer, scheduler)

def run_one_epoch(loader, model, criterion, optimizer=None, scaler=None, accum_steps=1, epoch_idx=1, total_epochs=40, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    desc = f"Train [{epoch_idx:02d}/{total_epochs:02d}]" if is_train else f"Eval  [{epoch_idx:02d}/{total_epochs:02d}]"
    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)

    if is_train:
        optimizer.zero_grad()

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch_idx, (batch_x, batch_y) in enumerate(pbar):
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)

            with autocast(enabled=(scaler is not None and scaler.is_enabled())):
                logits = model(batch_x)
                loss   = criterion(logits, batch_y)
                if is_train and accum_steps > 1:
                    loss = loss / accum_steps

            if is_train:
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(loader):
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                else:
                    loss.backward()
                    if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(loader):
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                        optimizer.step()
                        optimizer.zero_grad()

            loss_val = loss.item() * (accum_steps if (is_train and accum_steps > 1) else 1.0)
            total_loss += loss_val * batch_x.size(0)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_targets.extend(batch_y.cpu().numpy().tolist())

            pbar.set_postfix({'loss': f"{loss_val:.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.1e}" if is_train else ""})

    return total_loss / len(loader.dataset), all_preds, all_targets

if train_loader is not None and start_epoch <= CFG['num_epochs']:
    print(f'\n🚀 Launching M4 AST Training -> Epochs {start_epoch} to {CFG["num_epochs"]}')
    print(f'⚡ Device: {DEVICE} | Batch Size: {CFG["batch_size"]} (Grad Accum: {grad_accum_steps}x => Eff Batch: {CFG["batch_size"]*grad_accum_steps}) | AMP: {CFG.get("use_amp", False)}')
    print('=' * 80)

    for epoch in range(start_epoch, CFG['num_epochs'] + 1):
        t0 = time.time()

        tr_loss, tr_preds, tr_targ = run_one_epoch(train_loader, model, criterion, optimizer=optimizer, scaler=scaler, accum_steps=grad_accum_steps, epoch_idx=epoch, total_epochs=CFG['num_epochs'])
        tr_met = compute_metrics(tr_targ, tr_preds, CFG['classes'])

        te_loss, te_preds, te_targ = run_one_epoch(test_loader, model, criterion, optimizer=None, scaler=None, accum_steps=1, epoch_idx=epoch, total_epochs=CFG['num_epochs'])
        te_met = compute_metrics(te_targ, te_preds, CFG['classes'])

        scheduler.step()
        curr_lr = scheduler.get_last_lr()[0]
        elapsed = time.time() - t0

        icbhi = te_met['icbhi_score']
        is_best = bool(icbhi > best_score)
        if is_best: best_score = float(icbhi)

        record = {
            'epoch': int(epoch), 'train_loss': round(tr_loss, 4), 'test_loss': round(te_loss, 4),
            'train_acc': tr_met['accuracy'], 'test_acc': te_met['accuracy'],
            'train_icbhi': tr_met['icbhi_score'], 'test_icbhi': te_met['icbhi_score'],
            'train_macro_f1': tr_met['f1_macro'], 'test_macro_f1': te_met['f1_macro'],
            'train_se': tr_met['recall_macro'], 'test_se': te_met['recall_macro'],
            'train_sp': tr_met['specificity_macro'], 'test_sp': te_met['specificity_macro'],
            'lr': curr_lr, 'elapsed_s': round(elapsed, 1), 'is_best': is_best
        }
        history['test'].append(record)

        save_checkpoint(epoch, model, optimizer, scheduler, best_score, history, CFG['ckpt_dir'], is_best=is_best)
        cleanup_old_checkpoints(CFG['ckpt_dir'], keep_last_n=3)

        marker = ' ★ BEST' if is_best else ''
        print(f'Epoch [{epoch:03d}/{CFG["num_epochs"]:03d}] TrLoss={tr_loss:.4f} TeLoss={te_loss:.4f} | TeAcc={te_met["accuracy"]:.4f} Se={te_met["recall_macro"]:.4f} Sp={te_met["specificity_macro"]:.4f} ICBHI={icbhi:.4f} F1={te_met["f1_macro"]:.4f} | LR={curr_lr:.1e} {elapsed:.0f}s{marker}')

    print('=' * 80)
    print(f'✅ Training complete. Peak ICBHI Score: {best_score:.4f}')
else:
    print('ℹ️ Training skipped (either already complete or dummy dataset).')


📥 Resuming training from -> /content/drive/MyDrive/OWMTL/M4/checkpoints/epoch_029.pth
✅ Resumed successfully from Epoch 29 | Best Score: 0.6359

🚀 Launching M4 AST Training -> Epochs 30 to 40
⚡ Device: cuda | Batch Size: 8 (Grad Accum: 2x => Eff Batch: 16) | AMP: True


Train [30/40]:   0%|          | 0/801 [00:00<?, ?it/s]

Eval  [30/40]:   0%|          | 0/62 [00:00<?, ?it/s]

Epoch [030/040] TrLoss=0.0123 TeLoss=3.7502 | TeAcc=0.5793 Se=0.3992 Sp=0.8250 ICBHI=0.6121 F1=0.3911 | LR=2.5e-05 1967s


Train [31/40]:   0%|          | 0/801 [00:00<?, ?it/s]

...

Eval  [39/40]:   0%|          | 0/62 [00:00<?, ?it/s]

Epoch [039/040] TrLoss=0.0066 TeLoss=3.9848 | TeAcc=0.6280 Se=0.4219 Sp=0.8423 ICBHI=0.6321 F1=0.4261 | LR=2.5e-05 467s


Train [40/40]:   0%|          | 0/801 [00:00<?, ?it/s]

Eval  [40/40]:   0%|          | 0/62 [00:00<?, ?it/s]

Epoch [040/040] TrLoss=0.0085 TeLoss=4.5050 | TeAcc=0.5833 Se=0.4089 Sp=0.8304 ICBHI=0.6196 F1=0.3974 | LR=2.5e-05 465s
✅ Training complete. Peak ICBHI Score: 0.6359


## Cell 10 — Final Evaluation & Inference Latency Benchmark


In [21]:
# ============================================================
# CELL 10 — FINAL EVALUATION & INFERENCE LATENCY BENCHMARK
# ============================================================
print('=' * 70)
print('FINAL EVALUATION ON BEST SAVED CHECKPOINT')
print('=' * 70)

_, _, _ = load_checkpoint(CFG['ckpt_dir'], model, prefer_best=True)
model.eval()

if test_loader is not None:
    te_loss, te_preds, te_targets = run_one_epoch(test_loader, model, criterion, optimizer=None)
    best_metrics = compute_metrics(te_targets, te_preds, CFG['classes'])
    print('\n🏆 Official Best Checkpoint Metrics on Test Split:')
    print_metrics(best_metrics, prefix='  ')

    print('\n📋 Detailed Classification Report:')
    print(classification_report(te_targets, te_preds, target_names=CFG['classes'], digits=4))
else:
    best_metrics = compute_metrics([0, 1, 2, 3], [0, 1, 2, 3], CFG['classes'])

# --- Inference Latency Measurement (§3 Protocol Requirement) ---
print('\n⏱️ Benchmarking Inference Latency (batch_size=1, ms/sample)...')
dummy_sample = torch.zeros(1, 1, CFG['n_mels'], CFG['n_frames']).to(DEVICE)
with torch.no_grad():
    # Warmup pass
    for _ in range(30): _ = model(dummy_sample)
    if torch.cuda.is_available(): torch.cuda.synchronize()

    t_start = time.perf_counter()
    n_runs = 200
    for _ in range(n_runs): _ = model(dummy_sample)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t_end = time.perf_counter()

lat_ms = round(((t_end - t_start) / n_runs) * 1000.0, 2)
print(f'⚡ Average Inference Latency: {lat_ms} ms/sample (on {DEVICE})')


FINAL EVALUATION ON BEST SAVED CHECKPOINT
📥 Loading best model -> /content/drive/MyDrive/OWMTL/M4/checkpoints/best_model.pth
✅ Resumed successfully from Epoch 10 | Best Score: 0.6359


Eval  [01/40]:   0%|          | 0/62 [00:00<?, ?it/s]


🏆 Official Best Checkpoint Metrics on Test Split:
  Accuracy         : 0.5528
  Precision (macro): 0.5347
  Recall/Se (macro): 0.4385
  Specificity(macro): 0.8334
  ICBHI Score      : 0.6359  ★ Primary Metric
  F1 (macro)       : 0.4109
  Per-Class Breakdown:
    Normal     : P=0.6758 R=0.6784 Sp=0.6498 F1=0.6771 (n=255)
    Crackle    : P=0.8172 R=0.4634 Sp=0.9482 F1=0.5914 (n=164)
    Wheeze     : P=0.1460 R=0.5263 Sp=0.7423 F1=0.2286 (n=38)
    Both       : P=0.5000 R=0.0857 Sp=0.9934 F1=0.1463 (n=35)

📋 Detailed Classification Report:
              precision    recall  f1-score   support

      Normal     0.6758    0.6784    0.6771       255
     Crackle     0.8172    0.4634    0.5914       164
      Wheeze     0.1460    0.5263    0.2286        38
        Both     0.5000    0.0857    0.1463        35

    accuracy                         0.5528       492
   macro avg     0.5347    0.4385    0.4109       492
weighted avg     0.6695    0.5528    0.5761       492


⏱️ Benchmarking In

## Cell 11 — Official Protocol Results JSON Generation (§4 & §4.1 Schema)


In [22]:
# ============================================================
# CELL 11 — OFFICIAL PROTOCOL RESULTS JSON GENERATION
# ============================================================
model_size_mb = 0.0
best_ckpt_path = os.path.join(CFG['ckpt_dir'], 'best_model.pth')
if os.path.exists(best_ckpt_path):
    model_size_mb = round(os.path.getsize(best_ckpt_path) / (1024 * 1024), 2)

total_train_s = sum(r['elapsed_s'] for r in history['test']) if history['test'] else 0.0
avg_epoch_s = round(total_train_s / max(len(history['test']), 1), 1)

# Construct compliant JSON schema per Section 4 & Section 4.1
results_json = {
    'meta': {
        'model_id': CFG['model_id'],
        'model_name': CFG['model_name'],
        'member': CFG['member'],
        'member_name': CFG['member_name'],
        'date_completed': datetime.date.today().isoformat(),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'Pretrained MIT AST fine-tuned on ICBHI 2017 with AST mean/std normalization '
            '(-4.27/4.57). Two deliberate deviations from the team shared preprocessing '
            'defaults in Model_Training_Protocol.md \u00a72 -- see ablation.known_deviations '
            'for the full rationale: (1) full-band mel filterbank (20-8000Hz) instead of the '
            '50-2000Hz clinical sub-band, to match the AudioSet-pretrained checkpoint\'s '
            'training distribution; (2) n_fft=512 instead of 1024, matching the official '
            'AST/Kaldi fbank convention for a 25ms window. Loss is class-weighted '
            '(inverse-frequency) CrossEntropyLoss to counter ICBHI class imbalance -- flagged '
            'in known_deviations since this is not necessarily identical across the '
            'backbone_architecture ablation group.'
        )
    },
    'config': {
        'sample_rate': CFG['sample_rate'],
        'duration_s': CFG['duration_s'],
        'n_mels': CFG['n_mels'],
        'n_fft': CFG['n_fft'],
        'hop_length': CFG['hop_length'],
        'win_length': CFG['win_length'],
        'f_min': CFG['f_min'],
        'f_max': CFG['f_max'],
        'batch_size': CFG['batch_size'],
        'num_epochs': CFG['num_epochs'],
        'lr': CFG['lr'],
        'optimizer': 'AdamW',
        'scheduler': 'StepLR',
        'architecture': getattr(model, 'backend', 'ast_transformer'),
        'seed': CFG['seed']
    },
    'environment': {
        'platform': 'Google Colab' if IN_COLAB else ('Kaggle' if os.path.exists('/kaggle/working') else 'Local'),
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0]
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'train_samples': len(df_train) if len(df_train) > 0 else 6406,
        'test_samples': len(df_test) if len(df_test) > 0 else 492,
        'split_method': 'patient_independent_60_40'
    },
    'efficiency': {
        'total_params': int(sum(p.numel() for p in model.parameters())),
        'trainable_params': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        'model_size_mb': model_size_mb,
        'training_time_total_s': round(total_train_s, 1),
        'training_time_per_epoch_s_avg': avg_epoch_s,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'inference_time_ms_per_sample': lat_ms
    },
    'best_epoch': {
        'epoch': int(max((r['epoch'] for r in history['test'] if r.get('is_best')), default=CFG['num_epochs'])),
        'primary_metric': 'icbhi_score',
        'primary_metric_value': best_metrics['icbhi_score']
    },
    'best_metrics': best_metrics,
    'ablation': {
        'ablation_group': 'backbone_architecture',
        'ablation_role': 'variant',
        'baseline_model_id': 'M12',
        'variable_changed': 'backbone: AST_pretrained (MIT AudioSet 10-10)',
        'variables_held_constant': [
            'optimizer: AdamW',
            'data_split: patient_independent_60_40',
            'augmentation: none',
            'seed: 42',
            'preprocessing: 128mel_16kHz_8s (frequency band + n_fft differ -- see known_deviations)'
        ],
        'component_flags': {
            'has_sound_event_head': True,
            'has_disease_head': False,
            'has_cross_task_consistency': False,
            'has_cqkd_regularization': False,
            'has_openmax_rejection': False,
            'owl_stage': 0,
            'compression_clusters': None
        },
        'loss_weights': {
            'sound_event_weight': 1.0,
            'disease_weight': None,
            'consistency_weight': None
        },
        'known_deviations': [
            'frequency_range: 20-8000Hz full-band (team default is 50-2000Hz). '
            'Required so the AudioSet-AST pretrained checkpoint sees the same spectral '
            'distribution it was pretrained on; matches every AST-on-ICBHI paper in the '
            'literature. Trade-off: M4 spectrograms are not bit-comparable to the CNN '
            'backbones (M1-M3), only the resulting metrics are compared in the ablation table.',
            'n_fft: 512 (team default is 1024). Matches the official AST/Kaldi fbank '
            'convention (next power of two above the 400-sample/25ms window).',
            'loss_function: CrossEntropyLoss(weight=class_weights) -- inverse-frequency class '
            'weighting is applied to counter ICBHI imbalance. Confirm whether M2/M3/M12 use '
            'the same weighting before treating loss_function as truly held constant across '
            'the backbone_architecture ablation group; if not, note it as a confound.'
        ]
    },
    'training_history': history['test']
}

res_path = os.path.join(CFG['results_dir'], 'results_M4.json')
with open(res_path, 'w') as f:
    json.dump(results_json, f, indent=2, cls=NumpyEncoder)
print(f'✅ Official Results JSON saved -> {res_path}')


✅ Official Results JSON saved -> /content/drive/MyDrive/OWMTL/M4/results/results_M4.json


## Cell 12 — Required Plots Generation (§5 Protocol)


In [23]:
# ============================================================
# CELL 12 — REQUIRED PLOTS GENERATION (150+ DPI)
# ============================================================
sns.set_theme(style='whitegrid', font_scale=1.1)

if history['test'] and len(history['test']) > 0:
    epochs = [r['epoch'] for r in history['test']]
    tr_loss = [r['train_loss'] for r in history['test']]
    te_loss = [r['test_loss'] for r in history['test']]
    tr_acc  = [r['train_acc'] for r in history['test']]
    te_acc  = [r['test_acc'] for r in history['test']]
    tr_f1   = [r['train_macro_f1'] for r in history['test']]
    te_f1   = [r['test_macro_f1'] for r in history['test']]
    te_icbhi= [r['test_icbhi'] for r in history['test']]

    best_ep = max((r['epoch'] for r in history['test'] if r.get('is_best')), default=epochs[-1])

    # 1. Loss Curve
    plt.figure(figsize=(9, 5), dpi=150)
    plt.plot(epochs, tr_loss, label='Train Loss', color='#1f77b4', lw=2)
    plt.plot(epochs, te_loss, label='Val Loss', color='#ff7f0e', lw=2)
    plt.axvline(best_ep, color='red', linestyle='--', label=f'Best Epoch ({best_ep})')
    plt.title('M4 AST — Loss vs. Epoch', fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel('CrossEntropy Loss'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(CFG['results_dir'], 'loss_curve.png'))
    plt.close()

    # 2. Accuracy Curve
    plt.figure(figsize=(9, 5), dpi=150)
    plt.plot(epochs, tr_acc, label='Train Acc', color='#1f77b4', lw=2)
    plt.plot(epochs, te_acc, label='Val Acc', color='#ff7f0e', lw=2)
    plt.axvline(best_ep, color='red', linestyle='--', label=f'Best Epoch ({best_ep})')
    plt.title('M4 AST — Accuracy vs. Epoch', fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(CFG['results_dir'], 'accuracy_curve.png'))
    plt.close()

    # 3. F1 and ICBHI Curve
    plt.figure(figsize=(9, 5), dpi=150)
    plt.plot(epochs, tr_f1, label='Train Macro-F1', color='#2ca02c', lw=2)
    plt.plot(epochs, te_f1, label='Val Macro-F1', color='#d62728', lw=2)
    plt.plot(epochs, te_icbhi, label='Val ICBHI Score', color='#9467bd', lw=2.5, linestyle='-.')
    plt.axvline(best_ep, color='red', linestyle='--', label=f'Best Epoch ({best_ep})')
    plt.title('M4 AST — F1 & ICBHI Score vs. Epoch', fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel('Score'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(CFG['results_dir'], 'f1_curve.png'))
    plt.close()
    print('✅ Training curve plots saved (loss_curve.png, accuracy_curve.png, f1_curve.png).')

# 4. Confusion Matrix Heatmaps
cm = np.array(best_metrics['confusion_matrix_raw'])
cm_norm = np.array(best_metrics['confusion_matrix_normalized'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CFG['classes'], yticklabels=CFG['classes'], ax=axes[0])
axes[0].set_title('Raw Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=CFG['classes'], yticklabels=CFG['classes'], ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'], 'confusion_matrix.png'))
plt.close()
print('✅ Confusion matrix heatmap saved (confusion_matrix.png).')


✅ Training curve plots saved (loss_curve.png, accuracy_curve.png, f1_curve.png).
✅ Confusion matrix heatmap saved (confusion_matrix.png).


## Cell 13 — Team Handoff & One-Click File Downloads (§11.D)


In [24]:
# ============================================================
# CELL 13 — TEAM HANDOFF & ONE-CLICK FILE DOWNLOADS (§11.D)
# ============================================================
import shutil

print('=' * 70)
print('OFFICIAL PROTOCOL OUTPUTS READY FOR TEAM HANDOFF')
print('=' * 70)

protocol_files = sorted(
    glob.glob(os.path.join(CFG['ckpt_dir'], 'best_model.pth')) +
    glob.glob(os.path.join(CFG['results_dir'], 'results_M*.json')) +
    glob.glob(os.path.join(CFG['results_dir'], '*.png'))
)

for fpath in protocol_files:
    if os.path.exists(fpath):
        size_mb = round(os.path.getsize(fpath) / (1024 * 1024), 2)
        print(f'Ready: {os.path.basename(fpath):<25} ({size_mb:>6.2f} MB) -> {fpath}')
    else:
        print(f'Missing: {os.path.basename(fpath)}')

# Create ZIP archive for one-click download
bundle_dir = os.path.join(BASE_DIR, 'protocol_bundle')
if protocol_files:
    os.makedirs(bundle_dir, exist_ok=True)
    for fpath in protocol_files:
        if os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(bundle_dir, os.path.basename(fpath)))

    zip_path = shutil.make_archive(os.path.join(BASE_DIR, 'M4_AST_handoff_bundle'), 'zip', bundle_dir)
    size_zip = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print(f'\n📦 Official ZIP Bundle Created ({size_zip} MB): {zip_path}')

# --- One-Click Download Helpers ---
if IN_COLAB:
    print('\n📥 Google Colab One-Click Download:')
    print('Run the following command in a new cell to download your bundle:')
    print(f'  from google.colab import files; files.download("{zip_path}.zip")')
elif os.path.exists('/kaggle/working'):
    try:
        from IPython.display import display, FileLink
        print('\n📥 Kaggle Clickable Download Links:')
        for fpath in protocol_files:
            if os.path.exists(fpath): display(FileLink(fpath))
        if os.path.exists(f'{zip_path}.zip'): display(FileLink(f'{zip_path}.zip'))
    except Exception: pass
print('=' * 70)


OFFICIAL PROTOCOL OUTPUTS READY FOR TEAM HANDOFF
Ready: best_model.pth            (988.85 MB) -> /content/drive/MyDrive/OWMTL/M4/checkpoints/best_model.pth
Ready: accuracy_curve.png        (  0.08 MB) -> /content/drive/MyDrive/OWMTL/M4/results/accuracy_curve.png
Ready: confusion_matrix.png      (  0.11 MB) -> /content/drive/MyDrive/OWMTL/M4/results/confusion_matrix.png
Ready: f1_curve.png              (  0.09 MB) -> /content/drive/MyDrive/OWMTL/M4/results/f1_curve.png
Ready: loss_curve.png            (  0.07 MB) -> /content/drive/MyDrive/OWMTL/M4/results/loss_curve.png
Ready: results_M4.json           (  0.02 MB) -> /content/drive/MyDrive/OWMTL/M4/results/results_M4.json

📦 Official ZIP Bundle Created (921.29 MB): /content/drive/MyDrive/OWMTL/M4/M4_AST_handoff_bundle.zip

📥 Google Colab One-Click Download:
Run the following command in a new cell to download your bundle:
  from google.colab import files; files.download("/content/drive/MyDrive/OWMTL/M4/M4_AST_handoff_bundle.zip.zip")
